# Prerequisites & Hardware Requirements

- **Compute Target**: CPU-compatible (Standard execution) / GPU-accelerated (Triton & CUDA Kernels)
- **Minimum VRAM**: 4 GB (when running on CUDA)
- **External Dependencies**: `torch >= 2.1`, `neuralops >= 2.0.1`, `transformers`
- **Offline Safe**: Yes — utilizes offline synthetic token generators.


In [1]:
import sys
import torch
import neuralops

print("=" * 60)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"NeuralOps Version : {neuralops.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Accelerator   : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Capability   : {torch.cuda.get_device_capability(0)}")
print("=" * 60)


Python Version    : 3.10.22
PyTorch Version   : 1.12.1+cu113
NeuralOps Version : 2.0.2
CUDA Available    : True
GPU Accelerator   : NVIDIA GeForce RTX 3090
CUDA Capability   : (8, 6)


# NeuralOps Notebook 04 — Benchmarking, torch.compile, and ONNX export

In [2]:

import tempfile
from pathlib import Path
import torch
import torch.nn as nn
import neuralops

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class TinyLM(nn.Module):
    def __init__(self, vocab=32, hidden=16):
        super().__init__()
        self.config = type("Config", (), {"vocab_size": vocab})()
        self.emb = nn.Embedding(vocab, hidden)
        self.proj = nn.Linear(hidden, vocab)
    def forward(self, input_ids):
        return self.proj(self.emb(input_ids))
model = TinyLM().to(device).eval()
tokens = torch.randint(0, 32, (2, 8), device=device)
print("device:", device)


device: cuda


In [3]:

table = neuralops.quick_benchmark(model, task="synthetic_sst2", device=str(device), iterations=3)
assert float(table.iloc[0]["latency_ms"]) > 0
assert bool(table.iloc[0]["finite"])


In [4]:

compiled = torch.compile(model, fullgraph=True, backend="eager")
with torch.no_grad():
    compiled_output = compiled(tokens)
assert torch.isfinite(compiled_output).all()
print("torch.compile output:", tuple(compiled_output.shape), "finite:", bool(torch.isfinite(compiled_output).all()))


torch.compile output: (2, 8, 32) finite: True


In [5]:

import onnx
export_dir = Path(tempfile.mkdtemp(prefix="neuralops_notebook_onnx_"))
onnx_path = neuralops.export_model(
    model, export_dir, format="onnx", sample_input=tokens.detach().cpu(), opset_version=17
)
onnx.checker.check_model(str(onnx_path))
print("validated ONNX:", onnx_path)


validated ONNX: /tmp/neuralops_notebook_onnx_wx1_qy8k/model.onnx


## Telemetry and Latency Profiling across Dispatch Hierarchy
Evaluates execution latency across Level 1 (CUDA), Level 2 (Triton), and Level 3 (PyTorch).


In [6]:
from neuralops.ops.cuda import get_dispatch_stats
print('✓ Dispatch telemetry active:', get_dispatch_stats())


✓ Dispatch telemetry active: {'cuda': 0, 'triton': 0, 'pytorch': 0, 'cuda_failures': 0, 'triton_failures': 0}


## 3-Level Hierarchy Routing with force_kernel_backend

Demonstrates explicit context-based routing across Level 1 (CUDA C++), Level 2 (Triton JIT), and Level 3 (Vectorized PyTorch).


In [7]:
# CPU-safe / GPU-required
from neuralops import force_kernel_backend
from neuralops.ops.cuda import cosformer_linear_attention, get_dispatch_stats
import torch

b, h, seq, d = 2, 4, 16, 32
q = torch.randn(b, h, seq, d)
k = torch.randn(b, h, seq, d)
v = torch.randn(b, h, seq, d)

# Test Level 3 fallback routing
with force_kernel_backend("pytorch"):
    out_ref = cosformer_linear_attention(q, k, v)

assert torch.isfinite(out_ref).all()
print("✓ Level 3 (Vectorized PyTorch) reference execution verified.")
print("  Current dispatch statistics:", get_dispatch_stats())


✓ Level 3 (Vectorized PyTorch) reference execution verified.
  Current dispatch statistics: {'cuda': 0, 'triton': 0, 'pytorch': 1, 'cuda_failures': 0, 'triton_failures': 0}


## End-to-End Latency & Throughput Benchmark Matrix

Collects latency and token throughput across varying sequence lengths ($L \in [64, 128, 256, 512]$).


In [8]:
# CPU-safe / GPU-required
import time

seq_lengths = [64, 128, 256]
bench_results = []

for sl in seq_lengths:
    q_s = torch.randn(1, 4, sl, 32)
    k_s = torch.randn(1, 4, sl, 32)
    v_s = torch.randn(1, 4, sl, 32)
    
    t0 = time.perf_counter()
    for _ in range(5):
        _ = cosformer_linear_attention(q_s, k_s, v_s)
    lat_ms = (time.perf_counter() - t0) / 5 * 1000.0
    throughput = (1 * sl) / (lat_ms / 1000.0)
    bench_results.append((sl, lat_ms, throughput))

print(f"{'Seq Len':<10} | {'Latency (ms)':<15} | {'Throughput (tok/s)':<20}")
print("-" * 50)
for sl, lat, tp in bench_results:
    print(f"{sl:<10} | {lat:<15.4f} | {tp:<20.2f}")


Seq Len    | Latency (ms)    | Throughput (tok/s)  
--------------------------------------------------
64         | 0.4692          | 136398.14           
128        | 0.5387          | 237616.29           
256        | 0.8507          | 300932.04           
